In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

### Dataset Exploration

In [ ]:
# look at a single .mat file

mat_file = './data/QLAK-CA1-08.mat'
file = h5py.File(mat_file, 'r')
print(list(file.keys()))

In [ ]:
# For HDF5 .mat files (v7.3), fields like 'trace' are MATLAB cell arrays
# stored as arrays of object references. You must dereference each one.

# Example: unpack 'trace' — each element is a reference to an array
trace_refs = file['trace']

# Dereference each object reference to get the actual data
index = 0
ref = trace_refs[index][0]

# neural and shape
# (71866 data points * 515 neurons)
print(file[ref][:], '\n')
print(file[ref][:].shape)

In [ ]:
pos_refs = file['position']

index = 0
ref = pos_refs[index][0]

print(file[ref][:], '\n')
print(file[ref][:].shape)

In [ ]:
# geometry of the environment
blk_refs = file['blocked'][:][0]
blks = [file[blk_refs[i]][:]
        for i in range(len(blk_refs))]

print(f"{len(blks)} sessions")
for i, b in enumerate(blks):
    print(f"  Session {i:2d}: {b.flatten()}")

### Understand how many neurons are seen each day

In [ ]:
# Load traces for all 31 sessions
trace_refs = file['trace']
n_sessions = trace_refs.shape[0]

traces = []
for i in range(n_sessions):
    ref = trace_refs[i][0]
    traces.append(file[ref][:])

n_neurons = traces[0].shape[1]
print(f"Sessions: {n_sessions}, Total neurons: {n_neurons}")
print(f"Timepoints per session: {[t.shape[0] for t in traces]}")

In [ ]:
# Build a presence matrix: (n_sessions x n_neurons)
# A neuron is "present" in a session if its column is NOT all-NaN
presence = np.zeros((n_sessions, n_neurons), dtype=bool)
for i, tr in enumerate(traces):
    presence[i] = ~np.all(np.isnan(tr), axis=0)

# How many neurons are active in each session
neurons_per_session = presence.sum(axis=1)
print("Neurons observed per session:")
for i, count in enumerate(neurons_per_session):
    print(f"  Session {i:2d}: {count:3d} / {n_neurons} neurons")

print(f"\nMin: {neurons_per_session.min()}, Max: {neurons_per_session.max()}, "
      f"Mean: {neurons_per_session.mean():.1f}")

In [ ]:
# How many sessions each neuron appears in
sessions_per_neuron = presence.sum(axis=0)

print("Distribution of neurons by number of sessions observed:")
for n_sess in range(0, n_sessions + 1):
    count = (sessions_per_neuron == n_sess).sum()
    if count > 0:
        print(f"  Seen in {n_sess:2d} sessions: {count:3d} neurons")

print(f"\nNeurons seen in ALL {n_sessions} sessions: {(sessions_per_neuron == n_sessions).sum()}")
print(f"Neurons seen in 0 sessions (always NaN): {(sessions_per_neuron == 0).sum()}")

### Convert to trial list format

In [ ]:
def mat_to_trials(filepath, n_blocked_positions=9):
    """Convert a .mat file into a list of trials.

    Each trial is a dict with:
        'trace':    np.array of shape (n_neurons, n_timepoints), NaN for missing neurons
        'position': np.array of shape (2, n_timepoints)
        'blocked':  np.array of shape (n_blocked_positions,), one-hot encoding

    Args:
        filepath: path to the .mat file
        n_blocked_positions: number of possible blocked positions (default 9)

    Returns:
        list of trial dicts, one per session
    """
    f = h5py.File(filepath, 'r')

    trace_refs = f['trace']
    pos_refs = f['position']
    blk_refs = f['blocked'][:][0]

    n_sessions = trace_refs.shape[0]
    trials = []

    for i in range(n_sessions):
        # trace: (timepoints, neurons) -> (neurons, timepoints)
        trace = f[trace_refs[i][0]][:].T

        # position: (timepoints, 2) -> (2, n_timepoints)
        position = f[pos_refs[i][0]][:].T

        # blocked: indices -> one-hot, -1 means no blocked positions
        blk_indices = f[blk_refs[i]][:].flatten()
        blocked = np.zeros(n_blocked_positions, dtype=np.float64)
        if not (len(blk_indices) == 1 and blk_indices[0] == -1):
            blocked[blk_indices.astype(int)] = 1.0

        trials.append({
            'trace': trace,
            'position': position,
            'blocked': blocked,
        })

    f.close()
    return trials

In [ ]:
# Test it
trials = mat_to_trials('./data/QLAK-CA1-08.mat')

print(f"Number of trials: {len(trials)}")
for i, t in enumerate(trials):
    n_nan = np.all(np.isnan(t['trace']), axis=1).sum()
    print(f"  Trial {i:2d}: trace {t['trace'].shape} ({n_nan} missing neurons), "
          f"position {t['position'].shape}, "
          f"blocked {t['blocked'].astype(int)}")

In [ ]:
position = trials[0]['position']

plt.figure(figsize=(6, 6))
plt.scatter(position[0], position[1], s=5, alpha=0.20)
plt.title('Animal Position Over Time (Trial 0)')
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.show()

### Convert all subjects to final format

In [ ]:
import glob

mat_files = sorted(glob.glob('./data/*.mat'))
print(f"Found {len(mat_files)} subjects:\n")

for mat_file in mat_files:
    name = mat_file.split('/')[-1].replace('.mat', '')
    f = h5py.File(mat_file, 'r')

    trace_refs = f['trace']
    pos_refs = f['position']
    blk_refs = f['blocked'][:][0]
    n_trials = trace_refs.shape[0]
    n_neurons = f[trace_refs[0][0]].shape[1]

    print(f"{name}: {n_trials} trials, {n_neurons} neurons")
    for i in range(n_trials):
        tr = f[trace_refs[i][0]]
        pos = f[pos_refs[i][0]]
        n_nan = np.all(np.isnan(tr[:]), axis=0).sum()

        blk_indices = f[blk_refs[i]][:].flatten()
        blocked = np.zeros(9, dtype=int)
        if not (len(blk_indices) == 1 and blk_indices[0] == -1):
            blocked[blk_indices.astype(int)] = 1

        print(f"  Trial {i:2d}: trace {(n_neurons, tr.shape[0])} ({n_nan} missing neurons), "
              f"position {(2, pos.shape[0])}, "
              f"blocked {blocked}")

    f.close()
    print()

### Discretize position into grid classes

In [ ]:
def discretize_position(position, n_grid=3, arena_size=75.0):
    """Discretize continuous 2D position into grid class labels.

    The arena is a square of arena_size x arena_size (cm). Position is binned
    into an n_grid x n_grid grid, producing class labels 0..(n_grid^2 - 1)
    in row-major order (row 0 = bottom of arena).

    Args:
        position: (2, n_timepoints) array of (x, y) coordinates in cm
        n_grid: number of bins per dimension (default 3 -> 9 classes)
        arena_size: side length of the square arena in cm (default 75.0)

    Returns:
        (n_timepoints,) array of integer class labels
    """
    edges = np.linspace(0, arena_size, n_grid + 1)[1:-1]

    x_bin = np.clip(np.digitize(position[0], edges), 0, n_grid - 1)
    y_bin = np.clip(np.digitize(position[1], edges), 0, n_grid - 1)

    return (y_bin * n_grid + x_bin).astype(int)

In [ ]:
# Test on trial 0 and trial 1
for trial_idx in [0, 1]:
    pos = trials[trial_idx]['position']  # (2, n_timepoints)

    for n_grid in [3, 5]:
        classes = discretize_position(pos, n_grid=n_grid)
        unique, counts = np.unique(classes, return_counts=True)
        n_unassigned = np.sum((classes < 0) | (classes >= n_grid**2))
        print(f"Trial {trial_idx}, {n_grid}x{n_grid} grid ({n_grid**2} classes):")
        print(f"  shape: {classes.shape}, classes present: {len(unique)}/{n_grid**2}")
        print(f"  unassigned points: {n_unassigned} / {len(classes)}")
        print(f"  distribution: {dict(zip(unique, counts))}\n")

# Visualize: overlay grid on position scatter for trial 0
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
pos = trials[0]['position']

for ax, n_grid in zip(axes, [3, 5]):
    classes = discretize_position(pos, n_grid=n_grid)
    n_classes = n_grid ** 2
    cmap = plt.colormaps['nipy_spectral'].resampled(n_classes)
    sc = ax.scatter(pos[0], pos[1], c=classes, s=1, alpha=0.3,
                    cmap=cmap, vmin=-0.5, vmax=n_classes - 0.5)
    # draw grid lines
    for edge in np.linspace(0, 75, n_grid + 1):
        ax.axvline(edge, color='red', linewidth=0.5, linestyle='--')
        ax.axhline(edge, color='red', linewidth=0.5, linestyle='--')
    ax.set_xlim(0, 75)
    ax.set_ylim(0, 75)
    ax.set_aspect('equal')
    ax.set_title(f'Trial 0 — {n_grid}x{n_grid} grid ({n_classes} classes)')
    ax.set_xlabel('X (cm)')
    ax.set_ylabel('Y (cm)')
    plt.colorbar(sc, ax=ax, ticks=range(n_classes), label='Class')

plt.tight_layout()
plt.show()